## How is Michelin recognition distributed across global dining markets, and how does it vary by geography, cuisine, price level, and sustainability recognition?

Check location

In [0]:
SELECT
    restaurant_name,
    address,
    location,
    latitude,
    longitude
FROM workspace.michelin_analytics.silver_michelin_restaurants
ORDER BY RAND()
LIMIT 30;

restaurant_name,address,location,latitude,longitude
Y Talbot,"Tregaron, SY25 6JL, United Kingdom","Tregaron, United Kingdom",52.2194683,-3.9326439
CASAeBOTTEGA,"Piazza Giuseppe Garibaldi 2, Dolceacqua, 18035, Italy","Dolceacqua, Italy",43.8507484,7.6228987
La Table 101,"101 rue Moncey, Lyon, 69003, France","Lyon, France",45.7617397,4.8501681
Novo Maite Anduriña,"Plaza San Pedro Nolasco 5, Saragossa, 50001, Spain","Saragossa, Spain",41.6529178,-0.8768169
Les Orfèvres,"14 rue des Orfèvres, Amiens, 80000, France","Amiens, France",49.8958267,2.2991222
Al Gatto Verde,"Stradello Bonaghino 56, Modena, 41126, Italy","Modena, Italy",44.6102868,10.980026
L'Arpège,"22 place de Verdun, Tarbes, 65000, France","Tarbes, France",43.2335046,0.0743525
Fogón Cocina de Viñedo,"San Martín 1745, Mendoza, M5507, Argentina","Mendoza, Argentina",-33.0246477,-68.8725424
Chuan Tian Xia,"5502 Seventh Ave., Brooklyn, NY, 11220, USA","Brooklyn, NY, USA",40.6396599,-74.0090575
Wagyu Kappo Yoshida,"Unit 3-48, 48F, Naza Tower, Platinum Park, 10 Persiaran KLCC, Kuala Lumpur, 50088, Malaysia","Kuala Lumpur, Malaysia",3.1546359,101.7186231


In [0]:
SELECT
    location,
    COUNT(*) AS restaurant_count
FROM workspace.michelin_analytics.silver_michelin_restaurants
WHERE SIZE(SPLIT(location, ',')) = 1
GROUP BY location
ORDER BY restaurant_count DESC;

location,restaurant_count
Singapore,307
Dubai,106
Macau,61
Abu Dhabi,54
Luxembourg,20


# Gold Layer: Michelin Restaurant Analytics

This notebook creates the analytics-ready Gold dataset used for reporting and Power BI.

The Gold layer:
- Preserves restaurant-level granularity
- Standardizes geographic dimensions
- Retains Michelin award and sustainability metrics
- Provides standardized price tiers
- Supports analysis by location, cuisine, price, award, and Green Star status

In [0]:
CREATE OR REPLACE TABLE workspace.michelin_analytics.gold_restaurant_analytics AS

SELECT
    restaurant_id,
    restaurant_name,

    -- Geography
    location,

    CASE
        WHEN location = 'Singapore' THEN 'Singapore'
        WHEN location = 'Dubai' THEN 'Dubai'
        WHEN location = 'Macau' THEN 'Macau'
        WHEN location = 'Abu Dhabi' THEN 'Abu Dhabi'
        WHEN location = 'Luxembourg' THEN 'Luxembourg'
        ELSE TRIM(SPLIT(location, ',')[0])
    END AS city,

    CASE
        WHEN location = 'Singapore' THEN 'Singapore'
        WHEN location IN ('Dubai', 'Abu Dhabi') THEN 'United Arab Emirates'
        WHEN location = 'Macau' THEN 'Macau'
        WHEN location = 'Luxembourg' THEN 'Luxembourg'
        ELSE TRIM(ELEMENT_AT(SPLIT(location, ','), -1))
    END AS country,

    latitude,
    longitude,

    -- Restaurant attributes
    cuisine,

    price_tier,

    CASE
        WHEN price_tier = 1 THEN 'Price Tier 1'
        WHEN price_tier = 2 THEN 'Price Tier 2'
        WHEN price_tier = 3 THEN 'Price Tier 3'
        WHEN price_tier = 4 THEN 'Price Tier 4'
        ELSE 'Not Available'
    END AS price_tier_label,

    award,
    star_count,
    is_starred,
    is_green_star,

    -- Additional information
    address,
    website_url,
    michelin_url

FROM workspace.michelin_analytics.silver_michelin_restaurants;

num_affected_rows,num_inserted_rows


### Validate

In [0]:
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT restaurant_id) AS unique_restaurants,
    COUNT(DISTINCT city) AS cities,
    COUNT(DISTINCT country) AS countries,
    COUNT(DISTINCT cuisine) AS cuisines,
    SUM(CASE WHEN country IS NULL OR TRIM(country) = '' THEN 1 ELSE 0 END)
        AS missing_country,
    SUM(CASE WHEN city IS NULL OR TRIM(city) = '' THEN 1 ELSE 0 END)
        AS missing_city
FROM workspace.michelin_analytics.gold_restaurant_analytics;

total_rows,unique_restaurants,cities,countries,cuisines,missing_country,missing_city
19622,19622,6113,52,267,0,0


In [0]:
SELECT
    price_tier,
    price_tier_label,
    COUNT(*) AS restaurant_count
FROM workspace.michelin_analytics.gold_restaurant_analytics
GROUP BY
    price_tier,
    price_tier_label
ORDER BY price_tier;

price_tier,price_tier_label,restaurant_count
null,Not Available,1
1,Price Tier 1,1677
2,Price Tier 2,7824
3,Price Tier 3,6094
4,Price Tier 4,4026


Export Gold from Databricks

In [0]:
SELECT *
FROM workspace.michelin_analytics.gold_restaurant_analytics;

restaurant_id,restaurant_name,location,city,country,latitude,longitude,cuisine,price_tier,price_tier_label,award,star_count,is_starred,is_green_star,address,website_url,michelin_url
5684c10c3fd8fa406aaeb8c12fc138358ef5a6629c100400edd5732ba93fab4d,8 1/2 Otto e Mezzo - Bombana,"Hong Kong, Hong Kong SAR China",Hong Kong,Hong Kong SAR China,22.2814635,114.15867,Italian,4,Price Tier 4,3 Stars,3,true,false,"Shop 202, 2F, Alexandra House, 18 Chater Road, Central, Hong Kong, Hong Kong SAR China",https://www.ottoemezzobombana.com,https://guide.michelin.com/en/hong-kong-region/hong-kong/restaurant/8%C2%BD-otto-e-mezzo-bombana
7e7444d309ce34955ebc87ac74da9df7f74df156d10175f97223e95bc8e478bb,ABaC,"Barcelona, Spain",Barcelona,Spain,41.4104497,2.1365112,Creative,4,Price Tier 4,3 Stars,3,true,false,"Avenida del Tibidabo 1, Barcelona, 08022, Spain",https://abacrestaurant.com,https://guide.michelin.com/en/catalunya/barcelona/restaurant/abac
c8900b389c96cf16d10f5c4634f748d10ad0b93ac461f61ba6be3e57c07e5144,AM par Alexandre Mazzia,"Marseille, France",Marseille,France,43.2701104,5.3862325,Creative,4,Price Tier 4,3 Stars,3,true,false,"9 rue François-Rocca, Marseille, 13008, France",https://www.alexandre-mazzia.com/,https://guide.michelin.com/en/provence-alpes-cote-dazur/marseille/restaurant/am-par-alexandre-mazzia
1c16f60f4db9fd50a1ad802e44f5c7c740de3265388e96bb1247d12b7ceca7c1,Addison,"San Diego, CA, USA",San Diego,USA,32.9412972,-117.1988908,Contemporary,4,Price Tier 4,3 Stars,3,true,false,"Fairmont Grand Del Mar, 5200 Grand Del Mar Way, San Diego, CA, 92130, USA",https://www.addisondelmar.com/,https://guide.michelin.com/en/california/us-san-diego/restaurant/addison
1cadd5f0f8c50b6c1cdbce09232b2cd1d19aa9d74699f1c4ba71bd2983c87a43,Akelaŕe,"Donostia / San Sebastián, Spain",Donostia / San Sebastián,Spain,43.3076976,-2.0436352,Creative,4,Price Tier 4,3 Stars,3,true,false,"Paseo del Padre Orcolaga 56, Donostia / San Sebastián, 20008, Spain",https://akelarre.net/,https://guide.michelin.com/en/pais-vasco/es-donostia-san-sebastian/restaurant/akelare
6b2afd7d6b0dcec78d50d0dcbc640e98d59c9b3280422332432f38fdf19b2458,Alain Ducasse at The Dorchester,"London, United Kingdom",London,United Kingdom,51.5073381,-0.152575,French,4,Price Tier 4,3 Stars,3,true,false,"The Dorchester Hotel, 53 Park Lane, Mayfair, London, W1K 1QA, United Kingdom",https://www.alainducasse-dorchester.com/,https://guide.michelin.com/en/greater-london/london/restaurant/alain-ducasse-at-the-dorchester
cb346553511bfd7650c37adde6eb3c3079c4d413b90ba74d9a6c2c5960a8a132,Alléno Paris au Pavillon Ledoyen,"Paris, France",Paris,France,48.8660932,2.3164993,Creative,4,Price Tier 4,3 Stars,3,true,false,"8 avenue Dutuit, Paris, 75008, France",https://www.yannick-alleno.com/les-etablissements-du-groupe/alleno-paris,https://guide.michelin.com/en/ile-de-france/paris/restaurant/alleno-paris-au-pavillon-ledoyen
a4185edf02fd2fc6db99308de6bdc003dc9db8919f9a669689e820f854038226,Amador,"Vienna, Austria",Vienna,Austria,48.2539188,16.3584526,Creative,4,Price Tier 4,3 Stars,3,true,false,"Grinzinger Straße 86, Vienna, 1190, Austria",https://www.restaurant-amador.com/,https://guide.michelin.com/en/vienna/wien/restaurant/amador
eeedba0c684c09eacc3873ba828653592b92e3be537f081168290aa91ac3048c,Amber,"Hong Kong, Hong Kong SAR China",Hong Kong,Hong Kong SAR China,22.2805311,114.1577065,French Contemporary,4,Price Tier 4,3 Stars,3,true,true,"7F, Mandarin Oriental The Landmark , 15 Queen's Road Central, Central, Hong Kong, Hong Kong SAR China",https://www.mandarinoriental.com/en/hong-kong/the-landmark/dine/amber,https://guide.michelin.com/en/hong-kong-region/hong-kong/restaurant/amber569032
6afa7c537f759febf175f53c427e098e0427dc1b5ba510fd6158315bd613e08f,Aponiente,"El Puerto de Santa María, Spain",El Puerto de Santa María,Spain,36.6032543,-6.2164679,Creative,4,Price Tier 4,3 Stars,3,true,true,"Francisco Cossi Ochoa, El Puerto de Santa María, 11500, Spain",https://www.aponiente.com/,https://guide.michelin.com/en/andalucia/el-